# 🎙️ Voces del Simulador Causa Raíz · 100% gratis

Este cuaderno graba las voces de los personajes con **Chatterbox**, un modelo de voz de código abierto (licencia MIT), usando la GPU gratuita de Google Colab. **No necesita tarjeta ni pagos**: solo una cuenta de Google.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Luego ejecuta las celdas **en orden**, presionando ▶ a la izquierda de cada una. El avance se guarda en tu Google Drive (carpeta `CausaRaiz_voces`): si Colab se desconecta, vuelve a ejecutar las celdas 1, 2 y 3 y luego la que estabas usando, y el cuaderno continúa donde quedó.

In [ ]:
#@title 1 · Preparar: instalar el modelo y conectar Google Drive
#@markdown Tarda 2–4 minutos. **La primera vez, al terminar, reinicia la sesión** (menú *Entorno de ejecución → Reiniciar sesión*) y vuelve a ejecutar esta celda: es rápido y evita errores de versiones.
import os, torch
if torch.cuda.is_available():
    print('✓ GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠ NO HAY GPU. Ve a «Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4». Sin GPU es muchísimo más lento.')
!pip install -q chatterbox-tts
# torchvision de Colab queda incompatible con la versión de torch que instala Chatterbox (y el modelo no la usa)
!pip uninstall -y -q torchvision
from google.colab import drive
drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/CausaRaiz_voces'
os.makedirs(CARPETA, exist_ok=True)
print('✓ Listo. Los archivos se guardan en Google Drive → CausaRaiz_voces')

In [ ]:
#@title 2 · Cargar el guion
#@markdown Se descarga automáticamente desde el repositorio de GitHub. Si el repositorio es privado, deja la dirección vacía y se abrirá un botón para subir `scripts/colab/guion-voces.json`.
url_guion = 'https://raw.githubusercontent.com/bastianmenesesm/CausaRaiz/main/scripts/colab/guion-voces.json' #@param {type:"string"}
import json, os, urllib.request
from google.colab import files
RUTA_GUION = f'{CARPETA}/guion-voces.json'
cargado = False
if url_guion.strip():
    try:
        urllib.request.urlretrieve(url_guion.strip(), RUTA_GUION)
        cargado = True
        print('✓ Guion descargado desde GitHub')
    except Exception as e:
        print('⚠ No se pudo descargar el guion desde GitHub:', e)
if not cargado and not os.path.exists(RUTA_GUION):
    subido = files.upload()
    nombre = next(iter(subido))
    with open(RUTA_GUION, 'wb') as f:
        f.write(subido[nombre])
GUION = json.load(open(RUTA_GUION, encoding='utf-8'))
CLIPS = GUION['clips']
PERSONAJES = {}
for c in CLIPS:
    PERSONAJES.setdefault(c['personaje'], {'nombre': c['nombre'], 'genero': c['genero'], 'escenario': c['escenario']})
print(f'✓ {len(CLIPS)} audios por grabar · {len(PERSONAJES)} personajes')

In [ ]:
#@title 3 · Voces de referencia (dan el acento chileno y el timbre de cada personaje)
#@markdown **Dataset chileno de Google:** grabaciones públicas de chilenos y chilenas (OpenSLR 71). Se descarga solo (≈ 5 min la primera vez).
#@markdown
#@markdown **Mis propias grabaciones:** sube un audio de 10 a 20 segundos por personaje, hablando con naturalidad, con el nombre del personaje: `jonathan.m4a`, `rosa.mp3`, etc. (graba solo a personas que te den su permiso). Los personajes sin grabación usan el dataset.
fuente = 'Dataset chileno de Google (OpenSLR 71)' #@param ['Dataset chileno de Google (OpenSLR 71)', 'Mis propias grabaciones']
import glob, re, subprocess, zipfile, random
import numpy as np, soundfile as sf

DIR_DATOS = '/content/slr71'
DIR_REF = f'{CARPETA}/referencias'
os.makedirs(DIR_REF, exist_ok=True)
ESPEJOS = ['https://www.openslr.org/resources/71', 'https://us.openslr.org/resources/71', 'https://openslr.elda.org/resources/71', 'https://openslr.magicdatatech.com/resources/71']

def descargar(nombre):
    destino = f'/content/{nombre}'
    if os.path.exists(destino) and os.path.getsize(destino) > 1_000_000:
        return destino
    for base in ESPEJOS:
        r = subprocess.run(['wget', '-q', '-O', destino, f'{base}/{nombre}'])
        if r.returncode == 0 and os.path.getsize(destino) > 1_000_000:
            return destino
    raise RuntimeError(f'No se pudo descargar {nombre}. Revisa tu conexión y vuelve a ejecutar la celda.')

def preparar_dataset():
    if glob.glob(f'{DIR_DATOS}/**/*.wav', recursive=True):
        return
    os.makedirs(DIR_DATOS, exist_ok=True)
    for genero, nombre in [('f', 'es_cl_female.zip'), ('m', 'es_cl_male.zip')]:
        print('Descargando', nombre, '…')
        z = descargar(nombre)
        with zipfile.ZipFile(z) as zz:
            zz.extractall(f'{DIR_DATOS}/{genero}')
    for lic in glob.glob(f'{DIR_DATOS}/**/LICENSE*', recursive=True)[:1]:
        print('\nLicencia del dataset (debes dar crédito a sus autores):\n' + open(lic, errors='ignore').read()[:600])

def locutores():
    # Devuelve {id_locutor: {'genero': 'f'|'m', 'archivos': [rutas]}}
    out = {}
    for genero in ['f', 'm']:
        for ruta in glob.glob(f'{DIR_DATOS}/{genero}/**/*.wav', recursive=True):
            partes = os.path.basename(ruta).split('_')
            loc = '_'.join(partes[:2]) if len(partes) >= 3 else partes[0]
            out.setdefault(loc, {'genero': genero, 'archivos': []})['archivos'].append(ruta)
    return {k: v for k, v in out.items() if len(v['archivos']) >= 3}

def referencia_de_locutor(loc, info, segundos=14):
    destino = f'{DIR_REF}/{loc}.wav'
    if os.path.exists(destino):
        return destino
    archivos = sorted(info['archivos'], key=os.path.getsize, reverse=True)
    trozos, sr, total = [], None, 0.0
    for a in archivos:
        x, s = sf.read(a, dtype='float32')
        if x.ndim > 1:
            x = x.mean(axis=1)
        if sr is None:
            sr = s
        if s != sr:
            continue
        trozos += [x, np.zeros(int(sr * 0.3), dtype='float32')]
        total += len(x) / sr
        if total >= segundos:
            break
    sf.write(destino, np.concatenate(trozos), sr)
    return destino

def convertir_propia(ruta, pid):
    destino = f'{DIR_REF}/propia_{pid}.wav'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', ruta, '-ac', '1', '-ar', '24000', destino], check=True)
    return destino

preparar_dataset()
LOCUTORES = locutores()
fem = sorted(k for k, v in LOCUTORES.items() if v['genero'] == 'f')
mas = sorted(k for k, v in LOCUTORES.items() if v['genero'] == 'm')
print(f'✓ {len(fem)} locutoras y {len(mas)} locutores chilenos disponibles')

# Asignación automática: voces distintas dentro de cada caso (se guarda y puedes cambiarla en la celda 4)
RUTA_ASIG = f'{CARPETA}/asignacion.json'
ASIGNACION = json.load(open(RUTA_ASIG)) if os.path.exists(RUTA_ASIG) else {}
escenarios = sorted({p['escenario'] for p in PERSONAJES.values()})
for ei, esc in enumerate(escenarios):
    contadores = {'f': 0, 'm': 0}
    for pid, p in PERSONAJES.items():
        if p['escenario'] != esc:
            continue
        lista = fem if p['genero'] == 'f' else mas
        i = contadores[p['genero']]
        contadores[p['genero']] += 1
        if pid not in ASIGNACION and lista:
            ASIGNACION[pid] = lista[(i + ei * 3) % len(lista)]

if fuente == 'Mis propias grabaciones':
    print('Sube las grabaciones (nombre del archivo = personaje, por ejemplo jonathan.m4a):')
    for nombre, datos in files.upload().items():
        pid = os.path.splitext(os.path.basename(nombre))[0].lower()
        if pid not in PERSONAJES:
            print(f'  ⚠ {nombre}: no existe el personaje «{pid}». Personajes: {", ".join(PERSONAJES)}')
            continue
        tmp = f'/content/{nombre}'
        open(tmp, 'wb').write(datos)
        convertir_propia(tmp, pid)
        ASIGNACION[pid] = 'propia:' + pid
        print(f'  ✓ {PERSONAJES[pid]["nombre"]} usará tu grabación')

json.dump(ASIGNACION, open(RUTA_ASIG, 'w'), indent=1)

def ref_de(pid):
    v = ASIGNACION[pid]
    if v.startswith('propia:'):
        return f'{DIR_REF}/propia_{pid}.wav'
    return referencia_de_locutor(v, LOCUTORES[v])

for pid, p in PERSONAJES.items():
    print(f'  {p["escenario"]:<15} {p["nombre"]:<20} → {ASIGNACION.get(pid, "¿sin voz?")}')

In [ ]:
#@title 4 · (Opcional) Escuchar locutores y cambiar voces
#@markdown Marca **escuchar** para oír una muestra de cada locutor del dataset. Para cambiar voces escribe pares `personaje=locutor` separados por comas, por ejemplo: `luis=clm_08784, rosa=clf_05223`
escuchar = False #@param {type:"boolean"}
cambios = '' #@param {type:"string"}
from IPython.display import Audio, display, Markdown
if escuchar:
    for loc in fem + mas:
        display(Markdown(f'**{loc}** ({"mujer" if LOCUTORES[loc]["genero"] == "f" else "hombre"})'))
        display(Audio(referencia_de_locutor(loc, LOCUTORES[loc])))
for par in [x.strip() for x in cambios.split(',') if '=' in x]:
    pid, loc = [y.strip() for y in par.split('=', 1)]
    if pid not in PERSONAJES:
        print('⚠ personaje desconocido:', pid)
    elif loc not in LOCUTORES and not loc.startswith('propia:'):
        print('⚠ locutor desconocido:', loc)
    else:
        ASIGNACION[pid] = loc
        print(f'✓ {PERSONAJES[pid]["nombre"]} → {loc}')
json.dump(ASIGNACION, open(RUTA_ASIG, 'w'), indent=1)

In [ ]:
#@title 5 · Motor de voz (ejecútala una vez; carga el modelo, ~1 min)
import inspect, time, torch, torchaudio
from IPython.display import Audio, display, Markdown
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELO = ChatterboxMultilingualTTS.from_pretrained(device=DISPOSITIVO)
PARAMS = inspect.signature(MODELO.generate).parameters

def limpiar(t):
    t = re.sub(r'[«»“”"]', '', t).replace('…', '...')
    return re.sub(r'\s+', ' ', t).strip()

def frases(t, maximo=220, minimo=45):
    partes = [p.strip() for p in re.findall(r'[^.!?]+(?:\.\.\.|[.!?]+)?', t) if p.strip()]
    out = []
    for p in partes:
        if len(p) > maximo:
            out += [x.strip() for x in re.split(r'(?<=,)\s+', p) if x.strip()]
        elif out and len(out[-1]) < minimo and len(out[-1]) + len(p) < maximo:
            out[-1] += ' ' + p
        else:
            out.append(p)
    return out or [t]

def grabar(clip, salida_mp3):
    ref = ref_de(clip['personaje'])
    kw = dict(language_id='es', audio_prompt_path=ref, exaggeration=clip['exaggeration'], cfg_weight=clip['cfg_weight'], temperature=clip['temperature'])
    kw = {k: v for k, v in kw.items() if k in PARAMS}
    corto = torch.zeros(1, int(MODELO.sr * 0.22))
    largo = torch.zeros(1, int(MODELO.sr * 0.5))
    piezas = []
    for par in clip['parrafos']:
        for f in frases(limpiar(par)):
            w = MODELO.generate(f, **kw)
            w = w.detach().cpu().float()
            if w.dim() == 1:
                w = w.unsqueeze(0)
            piezas += [w, corto]
        piezas.append(largo)
    audio = torch.cat(piezas[:-1], dim=1)
    tmp = '/content/_tmp.wav'
    torchaudio.save(tmp, audio, MODELO.sr)
    os.makedirs(os.path.dirname(salida_mp3), exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-ac', '1', '-b:a', '64k', salida_mp3], check=True)

print('✓ Modelo cargado en', DISPOSITIVO)

In [ ]:
#@title 6 · Prueba: grabar y escuchar los saludos (≈ 23 audios, 5–10 min)
#@markdown Escucha cada personaje. Si alguno no te gusta, cámbialo en la celda 4 y vuelve a ejecutar esta.
from IPython.display import Audio, display, Markdown
DIR_PRUEBA = '/content/prueba'
os.makedirs(DIR_PRUEBA, exist_ok=True)
for c in [c for c in CLIPS if c['saludo']]:
    destino = f"{DIR_PRUEBA}/{c['clave'].replace('/', '__')}__{ASIGNACION[c['personaje']].replace(':', '-')}.mp3"
    if not os.path.exists(destino):
        grabar(c, destino)
    display(Markdown(f"**{c['nombre']}** · {c['escenario']} · ánimo: {c['animo']} · voz: {ASIGNACION[c['personaje']]}"))
    display(Audio(destino))

In [ ]:
#@title 7 · Grabar TODO (≈ 217 audios; 1 a 2 horas con GPU T4)
#@markdown Puedes dejarla corriendo. Si Colab se desconecta, ejecuta otra vez las celdas 1, 2, 3, 5 y esta: continúa donde quedó.
DIR_AUDIO = f'{CARPETA}/audio'
RUTA_MANIFEST = f'{DIR_AUDIO}/manifest.json'
os.makedirs(DIR_AUDIO, exist_ok=True)
manifest = json.load(open(RUTA_MANIFEST)) if os.path.exists(RUTA_MANIFEST) else {}

def pendiente(c):
    m = manifest.get(c['clave'])
    return not m or m.get('h') != c['h'] or m.get('voz') != ASIGNACION[c['personaje']] or not os.path.exists(f"{DIR_AUDIO}/{c['clave']}.mp3")

cola = [c for c in CLIPS if pendiente(c)]
print(f'{len(CLIPS) - len(cola)} listos · {len(cola)} por grabar')
inicio = time.time()
for i, c in enumerate(cola, 1):
    try:
        grabar(c, f"{DIR_AUDIO}/{c['clave']}.mp3")
        manifest[c['clave']] = {'h': c['h'], 'proveedor': 'chatterbox', 'voz': ASIGNACION[c['personaje']]}
        json.dump(dict(sorted(manifest.items())), open(RUTA_MANIFEST, 'w'), indent=1)
        resto = (time.time() - inicio) / i * (len(cola) - i)
        print(f'✓ [{i}/{len(cola)}] {c["clave"]} ({c["nombre"]}, {c["animo"]}) · faltan ≈ {resto / 60:.0f} min')
    except Exception as e:
        print(f'✖ [{i}/{len(cola)}] {c["clave"]}: {e}')
print('\nTerminado. Si hubo errores, ejecuta esta celda otra vez para reintentarlos.')

In [ ]:
#@title 8 · Descargar los audios (audio.zip)
import shutil
with open(f'{CARPETA}/audio/CREDITOS.txt', 'w') as f:
    f.write('Voces generadas con Chatterbox (Resemble AI, licencia MIT).\n')
    if any(not v.startswith('propia:') for v in ASIGNACION.values()):
        f.write('Voces de referencia: Crowdsourced high-quality Chilean Spanish speech data set, Google (OpenSLR 71). Ver su licencia en openslr.org/71.\n')
zipeado = shutil.make_archive('/content/audio', 'zip', f'{CARPETA}/audio')
print('✓', len(json.load(open(f'{CARPETA}/audio/manifest.json'))), 'audios en el zip')
files.download(zipeado)

## Último paso (en tu computador)

1. Descomprime `audio.zip` dentro de la carpeta del proyecto, en **`public/audio`** (debe quedar `public/audio/manifest.json` y las carpetas `ultima-milla`, `primera-milla` y `almacenamiento`).
2. Prueba con `npm run dev`.
3. En GitHub Desktop: **Commit** y **Push**. Vercel publica las voces automáticamente.